In [3]:
pip install pageindex 

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.2 MB 1.4 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.2 MB 1.5 MB/s eta 0:00:01
   ------------------------------------ --- 1.0/1.2 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.3 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install openai 


In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv('PAGEINDEX_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

print("pageindex key is loaded","yes" if PAGEINDEX_API_KEY else "missing")
print("openapi key is loaded","yes" if OPENAI_API_KEY else "missing")

pageindex key is loaded yes
openapi key is loaded yes


In [10]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key = PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key = OPENAI_API_KEY, base_url="https://openrouter.ai/api/v1")

In [3]:
## uploading the document

PDF_PATH = 'attention.pdf'

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"Uploaded!")
print(f" Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")


Uploading: attention.pdf
Uploaded!
 Document ID: pi-cmo1dzqxn0a3d01r4ff2igygg
   (Save this ID — you'll use it throughout the notebook)


In [4]:
print("Building tree index")
print(" (This document once per document - the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f" status: {status}")

    if status == "completed":
        print("\n tree index ready!")
        break
    elif status == "failed":
        print("\n processing failed. check your pdf format")
        break

    time.sleep(5)

Building tree index
 (This document once per document - the index is cached for reuse)
 status: processing
 status: processing
 status: processing
 status: completed

 tree index ready!


In [5]:
## fetch the full tree
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"top level section: {len(pageindex_tree)}")
print("\n raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

top level section: 1

 raw tree (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Attention Is All You Need\n\nAshish Vaswani*\nGoogle Brain\navaswani@google.com\n\nNoam Shazeer*\nGoogle Brain\nnoam@google.com\n\nNiki Parmar*\nGoogle Research\nnikip@google.com\n\nJakob Uszkoreit*\nGoogle Research\nusz@google.com\n\nLlion Jones*\nGoogle Research\nllion@google.com\n\nAidan N. Gomez*\u2020\nUniversity of Toronto\naidan@cs.toronto.edu\n\n\u0141ukasz Kaiser*\nGoogle Brain\nlukaszkaiser@google.com\n\nIllia Polosukhin*\u2020\nillia.polosukhin@gmail.com\n",
  "text": "# Attention Is All You Need\n\nAshish Vaswani*\nGoogle Brain\navaswani@google.com\n\nNoam Shazeer*\nGoogle Brain\nnoam@google.com\n\nNiki Parmar*\nGoogle Research\nnikip@google.com\n\nJakob Uszkoreit*\nGoogle Research\nusz@google.com\n\nLlion Jones*\nGoogle Research\nllion@google.com\n\nAidan N. Gomez*\u2020\nUniversity of Toronto\naidan@cs.toronto.edu\n\n\u0141

In [6]:
## print the full tree

def print_tree(nodes, indent=0):
    """recursivly print tree titles for a visual overview"""
    for node in nodes:
        prefix = " " * indent + ("|_" if indent > 0 else"")
        page  = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent +1)

print("full document structure:\n")

full document structure:



In [7]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total 

total = count_nodes(pageindex_tree)
print(f"total nodes in tree: {total}")
print("each node = one retrival section of the document")


total nodes in tree: 17
each node = one retrival section of the document


In [14]:
def llm_tree_search(query: str, tree: list, model:str ="openai/gpt-oss-120b:free") -> dict:
    """
    core pageindex retrival:
    sends the query + document tree to an LLM.
    LLm reasons over the structure and returns relevent node_ids.
    Robust RAG tree search - fixes ALL errors: IndexError, AttributeError, JSONDecodeError

    returns : dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """

    # compress tree to save tokens - only sends titles + short summeries
    def compress(nodes):
        out = []
        for n in nodes:
            entry={
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150] #first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)

    prompt = f""" you are given a query and a document's tree structure (like a table of contents).
    your task: identify which node IDs most likely contain the answer to the query.
    think step by step about which sections are relevent.

    Query: {query}

    Document Tree:
    {json.dumps(compressed_tree, indent=2)}

    reply ONLY in this exact JSON format:
    {{
        "thinking" : "<your step by step resoning>"
        "node_list": ["node_id1", "node_id2"]
    }}"""

    response = openai_client.chat.completions.create(
            model=model,
            messages=[{"role":"user", "content":prompt}],
            response_format={"type":"json_object"}
        )

    return json.loads(response.choices[0].message.content)

In [16]:
# Simple keyword fallback if tree search fails
def keyword_search(query, tree):
    terms = query.lower().split()
    def score_node(node):
        title = node['title'].lower()
        return sum(1 for t in terms if t in title)
    scored = [(score_node(n), n['node_id']) for n in flatten_tree(tree)]
    return {"node_list": [id for _, id in sorted(scored, reverse=True)[:3]]}


In [17]:
## test with the simple query
query = "What is the syllabus covered in encoder"

print(f" Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

 Query: What is the syllabus covered in encoder

LLM Reasoning:
The query asks for the syllabus (i.e., what is covered) of the encoder. In the document, the encoder is described primarily in the Model Architecture chapter. The subsection '3.1 Encoder and Decoder Stacks' (node 0005) directly outlines the encoder structure and therefore contains the syllabus. The parent node '3 Model Architecture' (node 0004) also gives a high‑level overview of the encoder‑decoder structure, which is relevant. Hence the most likely nodes are 0005 and, optionally, its parent 0004.

Selected Node IDs: ['0005', '0004']


In [25]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [ ]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b:free") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return " No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [27]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f" Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f" Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f" Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n Answer:\n{answer}")
    
    return answer

In [28]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the encoders?",
    tree=pageindex_tree
)

 Query: What are the encoders?

 Reasoning: The query asks for information about the encoders. In the paper, the encoder is described in the Model Architecture section and more specifically in the 'Encoder and Decoder Stacks' subsection. Theref...
 Retrieved node IDs: ['0004', '0005']
 Sections found: ['3 Model Architecture', '3.1 Encoder and Decoder Stacks']

 Answer:
The encoder is a stack of six identical layers. Each layer contains two sub‑layers: (1) a multi‑head self‑attention mechanism and (2) a position‑wise fully connected feed‑forward network. Both sub‑layers are wrapped with residual connections and layer‑normalization, and all sub‑layers (as well as the embedding layers) output vectors of dimension \(d_{\text{model}} = 512\) (Section ‘3.1 Encoder and Decoder Stacks’, Page 2).


In [30]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What are the syllabus covered in transformer?",
    "What are the syllabus covered in mulithead?",
    
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What are the syllabus covered in transformer?
A: The Transformer paper’s “syllabus” (i.e., the topics it covers) is organized into the following sections:

- **Model Architecture** – overview of the encoder‑decoder framework (Section ‘3 Model Architecture’, p. 2)  
- **Encoder and Decoder Stacks** – details of the six‑layer encoder and decoder, su...
-------------------------------------------------------

Q: What are the syllabus covered in mulithead?
A: The multi‑head attention module covers the following items:

* Linear projections of the queries, keys and values into \(h\) separate subspaces (each with its own learned matrices \(W_i^{Q}, W_i^{K}, W_i^{V}\)) (Section ‘3.2.2 Multi-Head Attention’, p. 4).  
* Independent attention computations on e...
-------------------------------------------------------
